# PART 4: CASE STUDIES

In this final part of the project, we conduct two different case studies: Firstly, we want to select the most promising new locations for wind farms based on the filtering done in part 3 of the project and based on our capacity factor models. Secondly, we will apply both our weather prediction models and our capacity factor prediction models on a Danish wind farm to make a power output prediction for the future and benchmark our models based on a prediction retrieved by webscraping.

# Selection of new locations

The first case study is the selection of new wind farm locations with high average power output out of a list of filtered locations. We do this by finding the similarity to each of our initial four wind farm locations for which we have already created our models. Then, for each new location, we apply the model corresponding to the most similar wind farm location in order to calculate the average capacity factor of the new location.

In [1]:
import pandas as pd
import requests

First, we have to read in the data frame of filtered locations from part 3.

In [2]:
filtered_locations = pd.read_csv("Potential_locations.csv")

In [3]:
filtered_locations

,LUCA_LUSE_,NUTS_CODE,ELEV,AREA,EAST,NORT,LON,LAT,land use data,wspd,proximity
0,U321,ITF22,57,2911.74122,4736000,2106000,14.989610,41.933872,Natural grassland and sparsely vegetated areas,15.916618,113.983765
1,U321,ITF34,463,2790.29343,4768000,1992000,15.284815,40.886273,Natural grassland and sparsely vegetated areas,16.127015,226.784890
2,U361,ITD54,161,2687.64663,4384000,2382000,10.791233,44.545930,Non-irrigated arable land,27.208003,124.832411
3,U340,ITG11,32,2464.87976,4542000,1646000,12.493474,37.860164,"Moors, heathland and sclerophyllous vegetation",14.472483,397.470487
4,U361,ITF61,193,6651.20382,4862000,1826000,16.244960,39.323859,Non-irrigated arable land,15.500963,1369.176660
5,U317,ITF43,118,2443.20211,4896000,1966000,16.767096,40.559915,"Land principally occupied by agriculture, with...",21.510412,1180.349091
6,U361,ITF34,561,2790.29343,4762000,1986000,15.209574,40.835902,Non-irrigated arable land,16.127015,614.800754
7,U313,ITF45,45,2759.85145,5018000,1906000,18.125690,39.911603,Complex cultivation patterns,15.554321,466.530636
8,U361,ITF65,103,3184.35686,4820000,1690000,15.656551,38.124646,Non-irrigated arable land,15.047971,1353.822437
9,U361,ITF41,578,7184.91037,4770000,2020000,15.329860,41.137544,Non-irrigated arable land,16.127015,344.351514


Before we start our predictions, we make a visualization: We plot all the onshore wind farms that are already operating in Italy, as well as the filtered locations. We want to see if the filtered locations are situated in areas where there are already several wind farms, in order to see if the filtering is reasonable. First, we plot the filtered locations on a map of Italy.

In [4]:
import folium
italy_map = folium.Map()

for i in range(len(filtered_locations)):
    folium.Marker([filtered_locations.iloc[i]['LAT'],filtered_locations.iloc[i]['LON']], radius=1).add_to(italy_map)
    
italy_map

Next, we read in a list of all the existing wind farms in the world and filter them by Italy, operating and onshore.

In [5]:
df = pd.read_excel('Global-Wind-Power-Tracker-January-2023.xlsx', sheet_name=1)
df

,Date Last Researched,Country,Project Name,Phase Name,Project Name in Local Language / Script,Other Name(s),Capacity (MW),Installation Type,Status,Start year,...,"Local area (taluk, county)","Major area (prefecture, district)",State/Province,Subregion,Region,GEM location ID,GEM phase ID,Other IDs (location),Other IDs (unit/phase),Wiki URL
0,2022-11-10 00:00:00,Algeria,Kabertene wind farm,NaN,"مزرعة رياح كبيرت,, مدينة أدرار",Kabartene wind farm,10.0,onshore,operating,2014.0,...,Tsabit District,NaN,Adrar,Northern Africa,Africa,L900124,G900162,NaN,NaN,https://gem.wiki/Kabertene_wind_farm
1,2022-11-10 00:00:00,Algeria,Khenchela wind farm,NaN,مزرعة رياح خنشلة,NaN,20.0,onshore,cancelled,NaN,...,NaN,NaN,NaN,Northern Africa,Africa,L900137,G900178,NaN,NaN,https://gem.wiki/Khenchela_wind_farm
2,2022-11-10 00:00:00,Algeria,Timimoun wind farm,NaN,NaN,Timimoun Sktm,50.0,unknown,cancelled,NaN,...,NaN,NaN,NaN,Northern Africa,Africa,L916594,G920620,NaN,NaN,https://gem.wiki/Timimoun_wind_farm
3,2022-11-10 00:00:00,Algeria,Upper Plateaus wind farm,NaN,NaN,NaN,5010.0,unknown,pre-construction,2030.0,...,NaN,NaN,NaN,Northern Africa,Africa,L900099,G900128,NaN,NaN,https://gem.wiki/Upper_Plateaus_wind_farm
4,2022-08-25 00:00:00,Angola,Benjamin wind farm,NaN,NaN,NaN,52.0,onshore,pre-construction,2028.0,...,NaN,NaN,Benguela Province,Sub-Saharan Africa,Africa,L916595,G920621,NaN,NaN,https://gem.wiki/Benjamin_wind_farm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21219,2022-07-19 00:00:00,Uruguay,Valentines wind farm,NaN,Parque Eólico Valentines,NaN,70.0,onshore,operating,2016.0,...,Valentines,NaN,Treinta y Tres Department,Latin America and the Caribbean,Americas,L904995,G906199,NaN,NaN,https://gem.wiki/Valentines_wind_farm
21220,2022-07-19 00:00:00,Uruguay,Villa Rodriguez wind farm,NaN,Parque Eólico Villa Rodriguez,NaN,10.0,onshore,operating,2016.0,...,NaN,NaN,San José,Latin America and the Caribbean,Americas,L905509,G907540,NaN,NaN,https://gem.wiki/Villa_Rodriguez_wind_farm
21221,2022-07-19 00:00:00,Venezuela,La Guajira wind farm,1-A,Parque Eólico La Guajira,NaN,12.0,onshore,mothballed,2013.0,...,Municipio Indígena Guajira,NaN,Zulia State,Latin America and the Caribbean,Americas,L905443,G907342,NaN,NaN,https://gem.wiki/La_Guajira_wind_farm
21222,2022-07-19 00:00:00,Venezuela,La Guajira wind farm,1-B,Parque Eólico La Guajira,NaN,12.0,onshore,mothballed,2013.0,...,Municipio Indígena Guajira,NaN,Zulia State,Latin America and the Caribbean,Americas,L905443,G907343,NaN,NaN,https://gem.wiki/La_Guajira_wind_farm


In [6]:
df_italy=df[df['Country']=='Italy']
df_italy=df_italy[df_italy['Status']=='operating']
df_italy=df_italy[df_italy['Installation Type']=='onshore']
df_italy.head()

,Date Last Researched,Country,Project Name,Phase Name,Project Name in Local Language / Script,Other Name(s),Capacity (MW),Installation Type,Status,Start year,...,"Local area (taluk, county)","Major area (prefecture, district)",State/Province,Subregion,Region,GEM location ID,GEM phase ID,Other IDs (location),Other IDs (unit/phase),Wiki URL
12387,2021-10-19 00:00:00,Italy,Acquaspruzza wind farm,NaN,NaN,NaN,24.0,onshore,operating,2009.0,...,Isernia,NaN,Molise,Southern Europe,Europe,L912316,G915412,NaN,NaN,https://gem.wiki/Acquaspruzza_wind_farm
12388,2021-10-19 00:00:00,Italy,Agrigento wind farm,NaN,NaN,NaN,33.0,onshore,operating,2007.0,...,NaN,Agrigento,NaN,Southern Europe,Europe,L913648,G916969,NaN,NaN,https://gem.wiki/Agrigento_wind_farm
12389,2021-10-19 00:00:00,Italy,Albareto wind farm,NaN,NaN,NaN,20.0,onshore,operating,2019.0,...,NaN,Parma,NaN,Southern Europe,Europe,L911441,G914421,NaN,NaN,https://gem.wiki/Albareto_wind_farm
12390,2021-10-19 00:00:00,Italy,Alberona (Fortore) wind farm,NaN,NaN,Toppo Seggio wind farm,26.0,onshore,operating,2008.0,...,Foggia,NaN,Apulia,Southern Europe,Europe,L912749,G915897,NaN,NaN,https://gem.wiki/Alberona_(Fortore)_wind_farm
12391,2021-10-19 00:00:00,Italy,Alberona (IVPC) wind farm,NaN,NaN,NaN,33.0,onshore,operating,1999.0,...,Foggia,NaN,Apulia,Southern Europe,Europe,L913587,G916896,NaN,NaN,https://gem.wiki/Alberona_(IVPC)_wind_farm


Now we can plot them against our filtered locations. We can see that, except for a few outliers in northern Italy, our filtered locations generally lie in areas where there are already many wind farms, which means that our filtering criteria probably coincide with those used by the wind power companies that built their wind farms in these regions.

In [7]:
for i in range(len(df_italy)):
    folium.CircleMarker([df_italy.iloc[i]['Latitude'],df_italy.iloc[i]['Longitude']], radius=1).add_to(italy_map)
    
italy_map

After doing these visualizations, we now actually want to select three of the possible locations based on our similarity measure and the capacity factor predictions.

The first step is to add a column containing the Köppen-Geiger climate classification (https://en.wikipedia.org/wiki/K%C3%B6ppen_climate_classification) of each of the locations. We will need this information for the similarity calculation that we will conduct later.

In [8]:
def get_koppen_climate_classification(latitude, longitude):
    
    url_curl = f'http://climateapi.scottpinkelman.com/api/v1/location/{latitude}/{longitude}'
    
    response = requests.get(url_curl)

    if response.status_code == 200:
        # Parse the response as JSON
        data = response.json()
        return data["return_values"][0]["koppen_geiger_zone"]
    else:
        print(f"Error: Received status code {response.status_code}")

In [9]:
filtered_locations["Climate"] = filtered_locations.apply(lambda row: get_koppen_climate_classification(row["LAT"], row["LON"]), axis=1)

For the similarity measure, we will only need the elevation, the average wind speed and the climate classification, the other columns in the data frame are only relevant for the filtering. For now, we also leave the latitude and longitude in, for easier application of later functions.

In [10]:
df_loc_info = filtered_locations.loc[:,["LAT", "LON", "ELEV", "wspd", "Climate"]]
df_loc_info

,LAT,LON,ELEV,wspd,Climate
0,41.933872,14.989610,57,15.916618,Cfa
1,40.886273,15.284815,463,16.127015,Csb
2,44.545930,10.791233,161,27.208003,Cfa
3,37.860164,12.493474,32,14.472483,Csa
4,39.323859,16.244960,193,15.500963,Csa
5,40.559915,16.767096,118,21.510412,Csa
6,40.835902,15.209574,561,16.127015,Csb
7,39.911603,18.125690,45,15.554321,Csa
8,38.124646,15.656551,103,15.047971,Csa
9,41.137544,15.329860,578,16.127015,Cfa


Now that we have prepared the data for the possible new locations, we still have to find the values for the same parameters also for our four initial wind farms. In order to be able to do this, we create a function that calls an API for getting the elevation at a given location, and also another function calculating the average wind speed over a given time period, which is based on Meteostat.

In [11]:
denmark_latitude = 57.0754
denmark_longitude = 10.0365
spain_latitude = 42.8233
spain_longitude = -9.0891
bulgaria_latitude = 43.375
bulgaria_longitude = 27.5897
greece_latitude = 38.3136
greece_longitude = 24.2047

In [12]:
def get_elevation(latitude, longitude):
    
    url_curl = f'https://api.opentopodata.org/v1/test-dataset?locations={latitude},{longitude}'
    response = requests.get(url_curl)

    if response.status_code == 200:
        # Parse the response as JSON
        data = response.json()
        return data["results"][0]["elevation"]
    else:
        print(f"Error: Received status code {response.status_code}")

In [13]:
from meteostat import Point, Hourly, Daily, Stations
from datetime import datetime

def calculate_average_wind_speed(lat, lon, start_date, end_date):

    # Get nearby weather stations
    stations = Stations()
    stations = stations.nearby(lat, lon)
    station = stations.fetch(1)

    # Query Meteostat API for daily wind data
    data = Daily(station, start_date, end_date).fetch()

    # Remove rows with missing wind speed values
    datadrop = data.dropna(subset=['wspd'])

    # Calculate average wind speed for the location
    avg_wind_speed = datadrop['wspd'].mean()

    return avg_wind_speed

Now we apply all the functions to our four initial wind farms, in order to get the desired data points.

In [14]:
denmark_elev = get_elevation(denmark_latitude, denmark_longitude)
denmark_wspd = calculate_average_wind_speed(denmark_latitude, denmark_longitude, datetime(2010, 1, 1), datetime(2022, 12, 31))
denmark_climate = get_koppen_climate_classification(denmark_latitude, denmark_longitude)
denmark_loc_info = [denmark_latitude, denmark_longitude, denmark_elev, denmark_wspd, denmark_climate]
denmark_loc_info

[57.0754, 10.0365, -1.3100420236587524, 17.19259748843354, 'Cfb']

In [15]:
spain_elev = get_elevation(spain_latitude, spain_longitude)
spain_wspd = calculate_average_wind_speed(spain_latitude, spain_longitude, datetime(2010, 1, 1), datetime(2022, 12, 31))
spain_climate = get_koppen_climate_classification(spain_latitude, spain_longitude)
spain_loc_info = [spain_latitude, spain_longitude, spain_elev, spain_wspd, spain_climate]
spain_loc_info

[42.8233, -9.0891, -59.17011642456055, 10.094385150812062, 'Csb']

In [16]:
bulgaria_elev = get_elevation(bulgaria_latitude, bulgaria_longitude)
bulgaria_wspd = calculate_average_wind_speed(bulgaria_latitude, bulgaria_longitude, datetime(2010, 1, 1), datetime(2022, 12, 31))
bulgaria_climate = get_koppen_climate_classification(bulgaria_latitude, bulgaria_longitude)
bulgaria_loc_info = [bulgaria_latitude, bulgaria_longitude, bulgaria_elev, bulgaria_wspd, bulgaria_climate]
bulgaria_loc_info

[43.375, 27.5897, 118.58306121826172, 12.260885608856078, 'Cfa']

In [17]:
greece_elev = get_elevation(greece_latitude, greece_longitude)
greece_wspd = calculate_average_wind_speed(greece_latitude, greece_longitude, datetime(2010, 1, 1), datetime(2022, 12, 31))
greece_climate = get_koppen_climate_classification(greece_latitude, greece_longitude)
greece_loc_info = [greece_latitude, greece_longitude, greece_elev, greece_wspd, greece_climate]
greece_loc_info

[38.3136, 24.2047, -131.28382873535156, 10.249931034482762, 'Csa']

In order to be able to create dummy variable for the climate column, we append the four data points to the data frame containing the potential locations.

In [18]:
df_loc_info.loc[len(df_loc_info)] = denmark_loc_info
df_loc_info.loc[len(df_loc_info)] = spain_loc_info
df_loc_info.loc[len(df_loc_info)] = bulgaria_loc_info
df_loc_info.loc[len(df_loc_info)] = greece_loc_info

In [19]:
df_loc_info

,LAT,LON,ELEV,wspd,Climate
0,41.933872,14.989610,57.000000,15.916618,Cfa
1,40.886273,15.284815,463.000000,16.127015,Csb
2,44.545930,10.791233,161.000000,27.208003,Cfa
3,37.860164,12.493474,32.000000,14.472483,Csa
4,39.323859,16.244960,193.000000,15.500963,Csa
5,40.559915,16.767096,118.000000,21.510412,Csa
6,40.835902,15.209574,561.000000,16.127015,Csb
7,39.911603,18.125690,45.000000,15.554321,Csa
8,38.124646,15.656551,103.000000,15.047971,Csa
9,41.137544,15.329860,578.000000,16.127015,Cfa


Now we create dummies for the climate column. The reason why we have to do that is that otherwise, we are not able to calculate our similarity measure, which cannot handle strings.

In [20]:
df_loc_info = pd.get_dummies(df_loc_info)

In [21]:
df_loc_info

,LAT,LON,ELEV,wspd,Climate_Cfa,Climate_Cfb,Climate_Csa,Climate_Csb,Climate_Dfb,Climate_ET
0,41.933872,14.989610,57.000000,15.916618,1,0,0,0,0,0
1,40.886273,15.284815,463.000000,16.127015,0,0,0,1,0,0
2,44.545930,10.791233,161.000000,27.208003,1,0,0,0,0,0
3,37.860164,12.493474,32.000000,14.472483,0,0,1,0,0,0
4,39.323859,16.244960,193.000000,15.500963,0,0,1,0,0,0
5,40.559915,16.767096,118.000000,21.510412,0,0,1,0,0,0
6,40.835902,15.209574,561.000000,16.127015,0,0,0,1,0,0
7,39.911603,18.125690,45.000000,15.554321,0,0,1,0,0,0
8,38.124646,15.656551,103.000000,15.047971,0,0,1,0,0,0
9,41.137544,15.329860,578.000000,16.127015,1,0,0,0,0,0


The last preparatory measure that we have to take is to normalize the elevation and the wind speed column, so that their values lie between 0 and 1 (such as those of the dummy variables, which we do not normalize).

In [22]:
from sklearn import preprocessing

# Normalize values
min_max_scaler = preprocessing.MinMaxScaler()
df_loc_info[['ELEV', 'wspd']] = min_max_scaler.fit_transform(df_loc_info[['ELEV', 'wspd']])

In [23]:
df_loc_info

,LAT,LON,ELEV,wspd,Climate_Cfa,Climate_Cfb,Climate_Csa,Climate_Csb,Climate_Dfb,Climate_ET
0,41.933872,14.989610,0.168670,0.340211,1,0,0,0,0,0
1,40.886273,15.284815,0.532377,0.352505,0,0,0,1,0,0
2,44.545930,10.791233,0.261836,1.000000,1,0,0,0,0,0
3,37.860164,12.493474,0.146274,0.255825,0,0,1,0,0,0
4,39.323859,16.244960,0.290503,0.315923,0,0,1,0,0,0
5,40.559915,16.767096,0.223316,0.667073,0,0,1,0,0,0
6,40.835902,15.209574,0.620168,0.352505,0,0,0,1,0,0
7,39.911603,18.125690,0.157920,0.319040,0,0,1,0,0,0
8,38.124646,15.656551,0.209878,0.289453,0,0,1,0,0,0
9,41.137544,15.329860,0.635397,0.352505,1,0,0,0,0,0


The next step is crucial: Now we calculate the most similar among our four initial wind farms for each of the possible new locations. We do this by applying a KDTree (k-dimensional tree) model. Essentially, this works by dividing the (in our case 8-dimensional) space into subspaces based on our four wind farm data points and then efficiently searching for the nearest neighbor for the new locations. As a consequence, each of the new locations gets assigned a label (0 to 3) that corresponds to one of the initial wind farms. These labels we then add as a new column to our data frame.

In [24]:
from scipy.spatial import KDTree
import numpy as np

np.random.seed(0)
df_loc_info_wind_farms = df_loc_info.iloc[(len(df_loc_info)-4) : (len(df_loc_info)), :]
df_loc_info_new_locations = df_loc_info.iloc[:(len(df_loc_info)-4), :]

kd_tree = KDTree(df_loc_info_wind_farms.iloc[:, 2:])
distances, labels = kd_tree.query(df_loc_info_new_locations.iloc[:, 2:])
print(labels)

[2 1 2 3 3 3 1 3 3 2 2 2 3 2 2 2 3 2]


In [25]:
df_loc_info_new_locations["Label"] = labels

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [26]:
df_loc_info_new_locations

,LAT,LON,ELEV,wspd,Climate_Cfa,Climate_Cfb,Climate_Csa,Climate_Csb,Climate_Dfb,Climate_ET,Label
0,41.933872,14.989610,0.168670,0.340211,1,0,0,0,0,0,2
1,40.886273,15.284815,0.532377,0.352505,0,0,0,1,0,0,1
2,44.545930,10.791233,0.261836,1.000000,1,0,0,0,0,0,2
3,37.860164,12.493474,0.146274,0.255825,0,0,1,0,0,0,3
4,39.323859,16.244960,0.290503,0.315923,0,0,1,0,0,0,3
5,40.559915,16.767096,0.223316,0.667073,0,0,1,0,0,0,3
6,40.835902,15.209574,0.620168,0.352505,0,0,0,1,0,0,1
7,39.911603,18.125690,0.157920,0.319040,0,0,1,0,0,0,3
8,38.124646,15.656551,0.209878,0.289453,0,0,1,0,0,0,3
9,41.137544,15.329860,0.635397,0.352505,1,0,0,0,0,0,2


Now we load the models from part 2.

In [27]:
from xgboost import XGBRegressor

xgb_denmark = XGBRegressor()
xgb_denmark.load_model("xgb_denmark.json")

xgb_spain = XGBRegressor()
xgb_spain.load_model("xgb_spain.json")

xgb_bulgaria = XGBRegressor()
xgb_bulgaria.load_model("xgb_bulgaria.json")

xgb_greece = XGBRegressor()
xgb_greece.load_model("xgb_greece.json")

We create a function that gives us the weather data at a specified location for the last three years. This will provide the input for the capacity factor prediction.

In [28]:
from meteostat import Point, Hourly, Stations
from datetime import datetime, timedelta

def get_weather_data_for_windfarm(latitude, longitude):

    # Get nearby weather stations
    stations = Stations()
    stations = stations.nearby(latitude, longitude)
    station = stations.fetch(1)

    # Set time period
    # year, month, day
    start = datetime.now()- timedelta(hours=24*365*3)
    end = datetime.now()

    # Get hourly data
    hourly_data = Hourly(station, start, end)
    hourly_data = hourly_data.fetch()
    
    return hourly_data

The next function is essential for this section: Taking a location together with its label (coming from the similarity calculation), it first applies the above weather data function. Then, based on the label, it selects one of the four XGBoost models trained in part 2. This model is then applied on the weather data for the last three years, which gives us a time series of capacity factors, from which we then calculate the average, in order to have a single number that we can compare to the other possible locations.

In [29]:
# Predict average capacity factor
def get_average_capacity_factor(latitude, longitude, label):
    
    # Get weather data for location for past three years
    weather_df = get_weather_data_for_windfarm(latitude, longitude)[['temp','wspd','pres']]
    
    # Normalize weather data
    min_max_scaler = preprocessing.MinMaxScaler()
    weather_df = min_max_scaler.fit_transform(weather_df)
    
    model = None
    
    # Select right model according to cluster
    if label == 0:
        model = xgb_denmark
    elif label == 1:
        model = xgb_spain
    elif label == 2:
        model = xgb_bulgaria
    elif label == 3:
        model = xgb_greece
    
    # Use selected model to predict capacity factors
    cap_facs = model.predict(weather_df)
    
    # Calculate average capacity factor over all three years
    av_cap_fac = np.mean(cap_facs)
    
    return av_cap_fac

Now we apply the above function to each of the rows in the data frame of possible locations.

In [32]:
df_loc_info_new_locations["Average Capacity Factor"] = df_loc_info_new_locations.apply(lambda row: get_average_capacity_factor(row["LAT"], row["LON"], row["Label"]), axis=1)

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [33]:
df_loc_info_new_locations

,LAT,LON,ELEV,wspd,Climate_Cfa,Climate_Cfb,Climate_Csa,Climate_Csb,Climate_Dfb,Climate_ET,Label,Average Capacity Factor
0,41.933872,14.989610,0.168670,0.340211,1,0,0,0,0,0,2,0.391066
1,40.886273,15.284815,0.532377,0.352505,0,0,0,1,0,0,1,0.367070
2,44.545930,10.791233,0.261836,1.000000,1,0,0,0,0,0,2,0.328884
3,37.860164,12.493474,0.146274,0.255825,0,0,1,0,0,0,3,0.481976
4,39.323859,16.244960,0.290503,0.315923,0,0,1,0,0,0,3,0.410657
5,40.559915,16.767096,0.223316,0.667073,0,0,1,0,0,0,3,0.475092
6,40.835902,15.209574,0.620168,0.352505,0,0,0,1,0,0,1,0.367070
7,39.911603,18.125690,0.157920,0.319040,0,0,1,0,0,0,3,0.364387
8,38.124646,15.656551,0.209878,0.289453,0,0,1,0,0,0,3,0.509018
9,41.137544,15.329860,0.635397,0.352505,1,0,0,0,0,0,2,0.365207


In the above data frame, we can now compare the different locations in Italy with regard to their average capacity factor. The ones with the highest value will probably provide the highest energy output if a wind farm is built there. We can see that the regions with the highest average capacity factor are 16, 8, 3 and 12, where the first one has the NUTS code ITF45, the second one ITF65 and the last two ITG11. Below, we plot these three regions (in red), which we recommend for a new wind farm in Italy. As we can see from the plot, all three locations are rather in southern Italy, which is plausible, since they are probably more exposed to wind than the north of Italy.

In [34]:
folium.Marker(location=[40.140930, 17.990291], popup="IF45", icon=folium.Icon(icon="star", color="red")).add_to(italy_map)
folium.Marker(location=[38.124646, 15.656551], popup="ITF65", icon=folium.Icon(icon="star", color="red")).add_to(italy_map)
folium.Marker(location=[37.860164, 12.493474], popup="ITG11", icon=folium.Icon(icon="star", color="red")).add_to(italy_map)
italy_map

# Application to existing wind farm for benchmarking

In the second case study that we conduct, we apply both our weather prediction and our capacity factor prediction models to an existing wind farm in Denmark, in order to predict its power output in the next four hours. We are selecting one of the wind farms for which we have retrieved data from The Wind Power webpage by web scraping. Namely, this webpage provides its own predictions of expected load rate (= capacity factor). Hence, we can use these data as a benchmark for our own model. For this case study, we are looking at the date 2023-04-05 and making a prediction for four hours on that day.

In [35]:
df_wind_farms = pd.read_csv("2023-04-05.csv")
df_wind_farms

,Wind farm name,Country,County / Zone,City,Total nominal power,Operational,Onshore wind farm,Expected load rate,Expected production,Latitude,Longitude,Geodetic system,Precise location,Google Maps view,Precise localization
0,Ajstrup (Vesthimmerland),Denmark,Vesthimmerland (Nordjylland),Ajstrup,225 kW,Operational,Onshore wind farm,0 %,0 kWh,NaN,NaN,NaN,NaN,NaN,NaN
1,Aker,Denmark,Bornholm (Hovedstaden),Aker,"1,575 kW",Operational,Onshore wind farm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ajstrup,Denmark,Aalborg (Nordjylland),Ajstrup,"1,800 kW",Operational,Onshore wind farm,6.7 %,"2,890 kWh","57° 11' 23.9""","10° 0' 35.9""",WGS84,yes,Google Maps view,no
3,Abildgard (Thisted),Denmark,Thisted (Nordjylland),Abildgard,600 kW,Operational,Onshore wind farm,0 %,0 kWh,NaN,NaN,NaN,NaN,NaN,NaN
4,Aeble,Denmark,Nyborg (Syddanmark),Aeble,750 kW,Operational,Onshore wind farm,7.5 %,"1,350 kWh","55° 15' 59.2""","10° 41' 2""",WGS84,NaN,NaN,NaN
5,Aker,Denmark,Bornholm (Hovedstaden),Aker,"1,575 kW",Operational,Onshore wind farm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Albæk (Frederikshavn),Denmark,Frederikshavn (Nordjylland),Albæk,750 kW,Operational,Onshore wind farm,NaN,NaN,"57° 36' 24.1""","10° 21' 39.3""",WGS84,NaN,NaN,no
7,Ajstrup,Denmark,Aalborg (Nordjylland),Ajstrup,"1,800 kW",Operational,Onshore wind farm,6.7 %,"2,890 kWh","57° 11' 23.9""","10° 0' 35.9""",WGS84,yes,Google Maps view,no
8,Abildgard (Thisted),Denmark,Thisted (Nordjylland),Abildgard,600 kW,Operational,Onshore wind farm,0 %,0 kWh,NaN,NaN,NaN,NaN,NaN,NaN
9,Adum,Denmark,Ringkobing-Skjern (Midtjylland),Adum,225 kW,NaN,Onshore wind farm,NaN,NaN,"55° 52' 29.8""","8° 35' 42.7""",WGS84,NaN,NaN,no


In [36]:
#!pip install lat_lon_parser

In [37]:
from lat_lon_parser import parse

We select the Ajstrup wind farm and convert its latitude and longitude to the desired format.

In [38]:
ajstrup_latitude = parse(df_wind_farms.loc[2, "Latitude"])
print(ajstrup_latitude)
ajstrup_longitude = parse(df_wind_farms.loc[2, "Longitude"])
print(ajstrup_longitude)

57.18997222222222
10.009972222222222


Then, reusing the functions from the previous section, we calculate the elevation, average wind speed and Köppen-Geiger classification for the location of the wind farm.

In [39]:
ajstrup_elev = get_elevation(ajstrup_latitude, ajstrup_longitude)
ajstrup_wspd = calculate_average_wind_speed(ajstrup_latitude, ajstrup_longitude, datetime(2010, 1, 1), datetime(2022, 12, 31))
ajstrup_climate = get_koppen_climate_classification(ajstrup_latitude, ajstrup_longitude)
ajstrup_loc_info = [ajstrup_latitude, ajstrup_longitude, ajstrup_elev, ajstrup_wspd, ajstrup_climate]
ajstrup_loc_info

[57.18997222222222,
 10.009972222222222,
 -14.04425048828125,
 17.19259748843354,
 'Cfb']

In the following, we conduct the same data preparation steps as in the previous section to prepare for the similarity calculation.

In [40]:
df_loc_info = pd.DataFrame(ajstrup_loc_info).T
df_loc_info.columns=["LAT", "LON", "ELEV", "wspd", "Climate"]
df_loc_info["LAT"] = df_loc_info["LAT"].astype(float)
df_loc_info["LON"] = df_loc_info["LON"].astype(float)
df_loc_info["ELEV"] = df_loc_info["ELEV"].astype(float)
df_loc_info["wspd"] = df_loc_info["wspd"].astype(float)
df_loc_info

,LAT,LON,ELEV,wspd,Climate
0,57.189972,10.009972,-14.04425,17.192597,Cfb


In [41]:
# append data for our four windfarm to data frame of filtered locations (so that we are able to create dummies)
df_loc_info.loc[len(df_loc_info)] = denmark_loc_info
df_loc_info.loc[len(df_loc_info)] = spain_loc_info
df_loc_info.loc[len(df_loc_info)] = bulgaria_loc_info
df_loc_info.loc[len(df_loc_info)] = greece_loc_info

In [42]:
df_loc_info

,LAT,LON,ELEV,wspd,Climate
0,57.189972,10.009972,-14.044250,17.192597,Cfb
1,57.075400,10.036500,-1.310042,17.192597,Cfb
2,42.823300,-9.089100,-59.170116,10.094385,Csb
3,43.375000,27.589700,118.583061,12.260886,Cfa
4,38.313600,24.204700,-131.283829,10.249931,Csa


In [43]:
# create dummies (because clustering cannot handle strings)
df_loc_info = pd.get_dummies(df_loc_info)

In [44]:
df_loc_info

,LAT,LON,ELEV,wspd,Climate_Cfa,Climate_Cfb,Climate_Csa,Climate_Csb
0,57.189972,10.009972,-14.044250,17.192597,0,1,0,0
1,57.075400,10.036500,-1.310042,17.192597,0,1,0,0
2,42.823300,-9.089100,-59.170116,10.094385,0,0,0,1
3,43.375000,27.589700,118.583061,12.260886,1,0,0,0
4,38.313600,24.204700,-131.283829,10.249931,0,0,1,0


In [45]:
from sklearn import preprocessing

# Normalize values
min_max_scaler = preprocessing.MinMaxScaler()
df_loc_info[['ELEV', 'wspd']] = min_max_scaler.fit_transform(df_loc_info[['ELEV', 'wspd']])

In [46]:
df_loc_info

,LAT,LON,ELEV,wspd,Climate_Cfa,Climate_Cfb,Climate_Csa,Climate_Csb
0,57.189972,10.009972,0.469208,1.000000,0,1,0,0
1,57.075400,10.036500,0.520172,1.000000,0,1,0,0
2,42.823300,-9.089100,0.288609,0.000000,0,0,0,1
3,43.375000,27.589700,1.000000,0.305218,1,0,0,0
4,38.313600,24.204700,0.000000,0.021913,0,0,1,0


Applying the same method as in the previous section (KDTree), we now find the most similar among the four initial wind farms to the Aajstrup wind farm. Not surprisingly, the similarity calculation delivers us the label 0, which corresponds to the Danish wind farm. This means that in four our predictions, we will later use the models that we trained on the Danish wind farm.

In [47]:
from scipy.spatial import KDTree
import numpy as np

np.random.seed(0)
df_loc_info_wind_farms = df_loc_info.iloc[(len(df_loc_info)-4) : (len(df_loc_info)), :]
df_loc_info_ajstrup = df_loc_info.iloc[:(len(df_loc_info)-4), :]

kd_tree = KDTree(df_loc_info_wind_farms.iloc[:, 2:])
distances, labels = kd_tree.query(df_loc_info_ajstrup.iloc[:, 2:])
print(labels)

[0]


In [48]:
df_loc_info_ajstrup["Label"] = labels

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


## Weather prediction 

In this subsection, we are preparing the models for weather prediction.

In [49]:
from prophet.serialize import model_to_json, model_from_json
import warnings
warnings.filterwarnings('ignore')

#### READING IN THE DATA

First of all, we load all our weather predictions models from part 1 (from the json files in which we have stored them).

In [50]:
# denmark models
with open('all_models_json/temp_model_denmark.json', 'r') as fin:
    temp_model_denmark = model_from_json(fin.read())  # Load model
with open('all_models_json/dwpt_model_denmark.json', 'r') as fin:
    dwpt_model_denmark = model_from_json(fin.read())  # Load model
with open('all_models_json/rhum_model_denmark.json', 'r') as fin:
    rhum_model_denmark = model_from_json(fin.read())  # Load model
with open('all_models_json/prcp_model_denmark.json', 'r') as fin:
    prcp_model_denmark = model_from_json(fin.read())  # Load model
with open('all_models_json/pres_model_denmark.json', 'r') as fin:
    pres_model_denmark = model_from_json(fin.read())  # Load model
with open('all_models_json/general_wind_model_denmark.json', 'r') as fin:
    general_wind_model_denmark = model_from_json(fin.read())  # Load model

In [51]:
# spain models
with open('all_models_json/temp_model_spain.json', 'r') as fin:
    temp_model_spain = model_from_json(fin.read())  # Load model
with open('all_models_json/dwpt_model_spain.json', 'r') as fin:
    dwpt_model_spain = model_from_json(fin.read())  # Load model
with open('all_models_json/rhum_model_spain.json', 'r') as fin:
    rhum_model_spain = model_from_json(fin.read())  # Load model
with open('all_models_json/prcp_model_spain.json', 'r') as fin:
    prcp_model_spain = model_from_json(fin.read())  # Load model
with open('all_models_json/pres_model_spain.json', 'r') as fin:
    pres_model_spain = model_from_json(fin.read())  # Load model
with open('all_models_json/general_wind_model_spain.json', 'r') as fin:
    general_wind_model_spain = model_from_json(fin.read())  # Load model

In [52]:
# bulgaria models
with open('all_models_json/temp_model_bulgaria.json', 'r') as fin:
    temp_model_bulgaria = model_from_json(fin.read())  # Load model
with open('all_models_json/dwpt_model_bulgaria.json', 'r') as fin:
    dwpt_model_bulgaria = model_from_json(fin.read())  # Load model
with open('all_models_json/rhum_model_bulgaria.json', 'r') as fin:
    rhum_model_bulgaria = model_from_json(fin.read())  # Load model
with open('all_models_json/prcp_model_bulgaria.json', 'r') as fin:
    prcp_model_bulgaria = model_from_json(fin.read())  # Load model
with open('all_models_json/pres_model_bulgaria.json', 'r') as fin:
    pres_model_bulgaria = model_from_json(fin.read())  # Load model
with open('all_models_json/general_wind_model_bulgaria.json', 'r') as fin:
    general_wind_model_bulgaria = model_from_json(fin.read())  # Load model

In [53]:
# greece2 models
with open('all_models_json/temp_model_greece2.json', 'r') as fin:
    temp_model_greece2 = model_from_json(fin.read())  # Load model
with open('all_models_json/dwpt_model_greece2.json', 'r') as fin:
    dwpt_model_greece2 = model_from_json(fin.read())  # Load model
with open('all_models_json/rhum_model_greece2.json', 'r') as fin:
    rhum_model_greece2 = model_from_json(fin.read())  # Load model
with open('all_models_json/prcp_model_greece2.json', 'r') as fin:
    prcp_model_greece2 = model_from_json(fin.read())  # Load model
with open('all_models_json/pres_model_greece2.json', 'r') as fin:
    pres_model_greece2 = model_from_json(fin.read())  # Load model
with open('all_models_json/general_wind_model_greece2.json', 'r') as fin:
    general_wind_model_greece2 = model_from_json(fin.read())  # Load model

#### HELPER METHODS

Next, we define auxiliary functions which will be used in the below weather prediction functions.

In [54]:
# reference : https://github.com/nicolasfauchereau/Auckland_Cycling/blob/master/code/utils.py
# For creating 'future' dataframe which is being used to generate future predictions
import pandas as pd
def add_regressor_to_future(future, regressors_df): 
    """
    adds extra regressors to a `future` DataFrame dataframe created by fbprophet
    Parameters
    ----------
    data : pandas.DataFrame
        A `future` DataFrame created by the fbprophet `make_future` method  
        
    regressors_df: pandas.DataFrame 
        The pandas.DataFrame containing the regressors (with a datetime index)
    Returns
    -------
    futures : pandas.DataFrame
        The `future` DataFrame with the regressors added
    """
    
    futures = future.copy() 
    
    futures.index = pd.to_datetime(futures.ds)
    
    regressors = pd.concat(regressors_df, axis=1)
    
    futures = futures.merge(regressors, left_index=True, right_index=True)
    
    futures = futures.reset_index(drop = True)
    
    return futures

In [55]:
# Extracts only the relevant attributes from the dataframe
def extract_forecast_data(forecast):
    predictions = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    return predictions

#### FORECASTING METHODS

The following functions are doing the actual weather predictions, based on already trained models. (See the comments in the respective cells for a description of what the respective function does.)

In [56]:
# this function accepts the model for which it creates a
# @future : future dataframe (next 4 hours)
# @forecast: model is fitter with the futuredataframe and creates a prediction for that time span
def forecast_model(model):
    future = model.make_future_dataframe(periods=4, freq='1h')
    forecast = model.predict(future)    
    return forecast

In [57]:
# When we choose a country e.g SPAIN, we predict each of the weather varaibles separately
# There is a model for each variable which predicts and returns the required column

def get_all_variables_predictions(temp_model, dwpt_model, rhum_model, prcp_model, pres_model):
    #temp
    forecast_temp = extract_forecast_data(forecast_model(temp_model))
    forecast_temp.rename(columns={'yhat': 'temp'}, inplace=True)
    forecast_temp.set_index('ds', inplace=True)
    forecast_temp = forecast_temp['temp']
    #dwpt
    forecast_dwpt = extract_forecast_data(forecast_model(dwpt_model))
    forecast_dwpt.rename(columns={'yhat': 'dwpt'}, inplace=True)
    forecast_dwpt.set_index('ds', inplace=True)
    forecast_dwpt = forecast_dwpt['dwpt']
    #rhum
    forecast_rhum = extract_forecast_data(forecast_model(rhum_model))
    forecast_rhum.rename(columns={'yhat': 'rhum'}, inplace=True)
    forecast_rhum.set_index('ds', inplace=True)
    forecast_rhum = forecast_rhum['rhum']
    #prcp
    forecast_prcp = extract_forecast_data(forecast_model(prcp_model))
    forecast_prcp.rename(columns={'yhat': 'prcp'}, inplace=True)
    forecast_prcp.set_index('ds', inplace=True)
    forecast_prcp = forecast_prcp['prcp']
    #pres
    forecast_pres = extract_forecast_data(forecast_model(pres_model))
    forecast_pres.rename(columns={'yhat': 'pres'}, inplace=True)
    forecast_pres.set_index('ds', inplace=True)
    forecast_pres = forecast_pres['pres']
    
    return forecast_temp, forecast_dwpt, forecast_rhum, forecast_prcp, forecast_pres

In [58]:
# This methods firstly calls the @get_all_variables_predictions to create univarate dataframe
# which will be used as a 'regressor' for the multivariate.
# Using this, we predict the wind in an multivariate way (prophet's VARIMA).
# In the end we return the full @df;(temp,dwpt,rhum,prcp,pres and wind for the next 4 hours)
def get_future_dataframe_with_regressors(general_wind_model, temp_model, dwpt_model, rhum_model, prcp_model, pres_model):
    #eg.: building future dataframe for multivariate
    future = general_wind_model.make_future_dataframe(periods=(4), freq='1h')
    
    #Predict UNIVARIATE
    temp_predictions, \
    dwpt_predictions, \
    rhum_predictions, \
    prcp_predictions, \
    pres_predictions = get_all_variables_predictions(temp_model, dwpt_model, rhum_model, prcp_model, pres_model)
    
    #Predict MULTIVARIATE
    futures = add_regressor_to_future(future, [temp_predictions, dwpt_predictions, rhum_predictions, prcp_predictions, pres_predictions])
    
    forecast_wind = extract_forecast_data(general_wind_model.predict(futures))
    forecast_wind.rename(columns={'yhat': 'wind'}, inplace=True)
    forecast_wind.set_index('ds', inplace=True)
    forecast_wind = forecast_wind['wind']
    
    df = pd.concat([forecast_wind, temp_predictions, dwpt_predictions, rhum_predictions, prcp_predictions, pres_predictions], axis=1)
    df.columns = ['wind', 'temp','dwpt','rhum','prcp','pres']
        
    return df


In [59]:
predicted_weather_ajstrup = get_future_dataframe_with_regressors(general_wind_model_denmark, temp_model_denmark, dwpt_model_denmark, rhum_model_denmark, prcp_model_denmark, pres_model_denmark)
predicted_weather_ajstrup = predicted_weather_ajstrup.iloc[len(predicted_weather_ajstrup)-4:,:]

In [60]:
predicted_weather_ajstrup

,wind,temp,dwpt,rhum,prcp,pres
ds,,,,,,
2023-04-05 05:00:00,3.118466,0.834860,-1.966643,77.418489,0.004620,1014.653218
2023-04-05 06:00:00,3.579739,1.303582,-1.669338,77.461426,0.002206,1014.749294
2023-04-05 07:00:00,4.446500,1.923169,-1.400225,77.494442,-0.000352,1014.848731
2023-04-05 08:00:00,5.670878,2.604483,-1.204270,77.517828,-0.003042,1014.951748


As a first benchmark, we retrieve the actual weather data for Ajstrup in that time period, in order to compare it to our predicted weather data.

In [61]:
weather_ajstrup = get_weather_data_for_windfarm(ajstrup_latitude, ajstrup_longitude)
weather_ajstrup = weather_ajstrup[weather_ajstrup.index <= datetime.strptime('2023-04-05 08:00:00', '%Y-%m-%d %H:%M:%S')]
weather_ajstrup = weather_ajstrup[weather_ajstrup.index >= datetime.strptime('2023-04-05 05:00:00', '%Y-%m-%d %H:%M:%S')]
weather_ajstrup = weather_ajstrup[['wspd', 'temp', 'dwpt', 'rhum', 'prcp', 'pres']]

In [62]:
weather_ajstrup

,wspd,temp,dwpt,rhum,prcp,pres
time,,,,,,
2023-04-05 05:00:00,5.4,-1.4,-2.1,95.0,0.0,1025.9
2023-04-05 06:00:00,2.0,0.7,-0.2,94.0,0.0,1025.9
2023-04-05 07:00:00,1.8,3.0,1.0,87.0,0.0,1025.9
2023-04-05 08:00:00,3.6,6.0,1.9,75.0,0.0,1025.6


As we can see, the weather prediction delivers values that are relatively close to the actual measurements, which is a good starting point for our further predictions.

## Combining WEATHER prediction with CAPACITY prediction

The following function is essential, since it delivers the final output of the case study. Based on the location of the wind farm (in our case Aajstrup) and the label that we have determined earlier (0 = Denmark), it selects and applies the corresponding models that have been trained in part 1 and 2 of the project. The output is then a the predicted capacity factors for the next four hours (in this case 2023-04-05 05:00:00 to 2023-04-05 08:00:00).

In [63]:
# Predict future capacity factors

def get_predicted_capacity_factors(latitude, longitude, label):
    
    min_max_scaler = preprocessing.MinMaxScaler()
    
    # Apply right models according to cluster
    if label == 0:
        weather_data = get_future_dataframe_with_regressors(general_wind_model_denmark, temp_model_denmark, dwpt_model_denmark, rhum_model_denmark, prcp_model_denmark, pres_model_denmark)
        weather_data = weather_data[['temp', 'wind', 'pres']]
        weather_data = min_max_scaler.fit_transform(weather_data)
        cap_facs = xgb_denmark.predict(weather_data)
    elif label == 1:
        weather_data = get_future_dataframe_with_regressors(general_wind_model_spain, temp_model_spain, dwpt_model_spain, rhum_model_spain, prcp_model_spain, pres_model_spain)
        weather_data = weather_data[['temp', 'wind', 'pres']]
        weather_data = min_max_scaler.fit_transform(weather_data)
        cap_facs = xgb_spain.predict(weather_data)
    elif label == 2:
        weather_data = get_future_dataframe_with_regressors(general_wind_model_bulgaria, temp_model_bulgaria, dwpt_model_bulgaria, rhum_model_bulgaria, prcp_model_bulgaria, pres_model_bulgaria)
        weather_data = weather_data[['temp', 'wind', 'pres']]
        weather_data = min_max_scaler.fit_transform(weather_data)
        cap_facs = xgb_bulgaria.predict(weather_data)
    elif label == 3:
        weather_data = get_future_dataframe_with_regressors(general_wind_model_greece2, temp_model_greece2, dwpt_model_greece2, rhum_model_greece2, prcp_model_greece2, pres_model_greece2)
        weather_data = weather_data[['temp', 'wind', 'pres']]
        weather_data = min_max_scaler.fit_transform(weather_data)
        cap_facs = xgb_greece.predict(weather_data)
    
    return cap_facs[len(weather_data)-4:]

In [64]:
# Get prediction of capacity factors for next four hours
capacity_factor_prediction = get_predicted_capacity_factors(df_loc_info_ajstrup.loc[0,"LAT"], df_loc_info_ajstrup.loc[0,"LON"], df_loc_info_ajstrup.loc[0,"Label"])

By applying the above function, we get the following prediction results, which we then add to our weather data frame.

In [65]:
capacity_factor_prediction

array([0.14195243, 0.17211784, 0.21814097, 0.24326707], dtype=float32)

In [66]:
predicted_weather_ajstrup = predicted_weather_ajstrup[['temp', 'wind', 'pres']]
predicted_weather_ajstrup["capacity factor"] = capacity_factor_prediction[len(capacity_factor_prediction)-4:]
predicted_weather_ajstrup

,temp,wind,pres,capacity factor
ds,,,,
2023-04-05 05:00:00,0.834860,3.118466,1014.653218,0.141952
2023-04-05 06:00:00,1.303582,3.579739,1014.749294,0.172118
2023-04-05 07:00:00,1.923169,4.446500,1014.848731,0.218141
2023-04-05 08:00:00,2.604483,5.670878,1014.951748,0.243267


Now we can compare the values that we got with the prediction retrieved by web scraping, which is a capacity factor of 6.7% on average on 2023-04-05. In comparison, our values, which only cover four hours of the day, lie between 14% and 25%. This means that our predictions lie significantly above the prediction given by "The Wind Power". However, this is still inside the expected range, since both our weather prediction models and our capacity factor model have errors, which sum up since we apply our models sequentially. These errors are hard to avoid in the scope of this project, for example due to missing computer power and limited domain knowledge. However, we have to state that our predictions still lie in a reasonable range from the benchmark we are using (which is actually also a prediction and can also have errors). This means that, within the scope of this project, we can be satisfied with the performance of our models.

## Conclusions

In conclusion, we can say that our models provide the user/customer with highly relevant insights on the power production of wind farms. These can both be used to predict promising new locations for wind farms and to predict the future output of a specific wind farm. The latter knowledge can further be applied to schedule energy-intensive processes of close-by company sites, which leads to increased sustainability and potentially reduced costs. Since we have trained several models on different sites throughout Europe and, furthermore, developed and tested a similarity measure, our models can easily be applied to both arbitrary existing wind farms, as well as new locations considered for building wind farms. By comparing our models' output with a benchmark, we additionally ensured that they deliver reasonable outputs. By utilizing higher computer power or increased knowledge on the different parameters that influence the output of a wind farm, our models can in the future still be improved, which is, however, out of the scope of this project.